## Simplest Introduction to PyTorch Lightning

> This notebook is a part of [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners). Visit the repository for complete set of learning materials.

### Neural Frameworks

In [Introduction to PyTorch](IntroPyTorch.ipynb) we wrote the training code ourselves: move data to the right device, loop through minibatches, compute the loss, call `zero_grad()`, `backward()` and `step()`, compute accuracy on validation data, and so on. This is great for understanding what is going on, and gives us full control, but we end up re-writing the same *boilerplate* in every project.

[PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) is a high-level API on top of PyTorch (in the same way as Keras is a high-level API on top of TensorFlow). It does not hide PyTorch from you - your model is still a `torch.nn.Module` and your tensors are still PyTorch tensors - but it organizes your code and takes care of the engineering details:

* the training / validation / test loops
* moving models and data to CPU, GPU or TPU
* logging of metrics, progress bars and saving checkpoints

In this notebook we will solve the **MNIST handwritten digits classification** problem, which is the task suggested at the end of the [Introduction to PyTorch](IntroPyTorch.ipynb) and [Introduction to Keras](IntroKeras.ipynb) notebooks. First we write a plain PyTorch training loop, and then we rewrite it with Lightning, to see how the two compare.

### Getting things ready

You need PyTorch, torchvision and PyTorch Lightning installed. If you followed the course setup, PyTorch and torchvision are already there. PyTorch Lightning is not part of the course environment, so install it by uncommenting and running the cell below (you can also use `conda install -c conda-forge pytorch-lightning`):

In [ ]:
# Uncomment to install PyTorch Lightning (only needed once)
# %pip install pytorch-lightning

In [ ]:
import torch
import torchvision
import pytorch_lightning as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f'PyTorch version = {torch.__version__}')
print(f'PyTorch Lightning version = {pl.__version__}')

pl.seed_everything(42) # fix random seeds (Python, NumPy and PyTorch) for reproducibility

## Sample Problem: MNIST Digits

**MNIST** is a dataset of 70,000 handwritten digits, each of them is a 28x28 grayscale image. Our task is to classify the image into one of 10 classes (digits `0`..`9`).

`torchvision` can download MNIST for us. We convert the images to tensors, and normalize them using the mean and standard deviation of the MNIST pixels (more about normalization in the [Keras notebook](IntroKeras.ipynb)).

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.1307,), (0.3081,))
])

mnist_full = torchvision.datasets.MNIST('data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST('data', train=False, download=True, transform=transform)

print(f'Training images: {len(mnist_full)}, test images: {len(mnist_test)}')

Let's look at a few images:

In [ ]:
fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    img, label = mnist_full[i]
    ax[i].imshow(img.squeeze(), cmap='gray')
    ax[i].set_title(label)
    ax[i].axis('off')
plt.show()

In machine learning we normally split the data into three parts:

* **training** dataset - used to adjust the weights of the network
* **validation** dataset - used to monitor the training after each epoch, and to detect [overfitting](README.md#overfitting)
* **test** dataset - used only once at the very end, to estimate the final quality of the model

MNIST already comes with the training/test split, so we only need to take a part of the training set for validation:

In [ ]:
train_ds, val_ds = torch.utils.data.random_split(
    mnist_full, [55000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64)
test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=64)

print(f'train: {len(train_ds)}, validation: {len(val_ds)}, test: {len(mnist_test)}')

## Starting Point: Training Loop in Plain PyTorch

Here is the network that we are going to use - a simple multi-layered perceptron with one hidden layer. It takes the 28x28 image, flattens it into a vector of 784 numbers, and outputs 10 numbers (**logits**, one for each digit):

In [ ]:
def create_net():
    return torch.nn.Sequential(
        torch.nn.Flatten(),
        torch.nn.Linear(28*28, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 10)
    )

print(create_net())

And here is the training function written in the way we did it in [Introduction to PyTorch](IntroPyTorch.ipynb). Pay attention to all the things we need to take care of ourselves:

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Doing computations on ' + device)

def train_pytorch(net, epochs=3, lr=1e-3):
    net = net.to(device)                                    # move model to the device
    optim = torch.optim.Adam(net.parameters(), lr=lr)       # choose optimizer
    for ep in range(epochs):
        net.train()                                         # switch to training mode
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)               # move data to the device
            loss = torch.nn.functional.cross_entropy(net(x), y)
            optim.zero_grad()                               # reset gradients
            loss.backward()                                 # compute gradients
            optim.step()                                    # update weights

        net.eval()                                          # switch to evaluation mode
        val_loss, correct = 0.0, 0
        with torch.no_grad():                               # we do not need gradients here
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                z = net(x)
                val_loss += torch.nn.functional.cross_entropy(z, y, reduction='sum').item()
                correct += (z.argmax(dim=1) == y).sum().item()
        print(f'Epoch {ep}: val loss = {val_loss/len(val_ds):.4f}, val acc = {correct/len(val_ds):.4f}')
    return net

net = train_pytorch(create_net())

It works, and the accuracy on the validation set should be around 96-97%. But if we also want to compute the accuracy on the test set, plot the training curves, save the best model, or run on several GPUs, we need to write even more code. Let's see how Lightning can help.

## Defining a Network as a LightningModule

In order to use Lightning, we organize our code into a class inherited from `pl.LightningModule` (which is itself a `torch.nn.Module`). The code that we have already written is not thrown away, it just goes to the places where Lightning expects it:

| What | Where it goes in `LightningModule` |
|------|------------------------------------|
| Layers of the network | `__init__` |
| Forward pass | `forward(self, x)` |
| Body of the training loop (loss computation) | `training_step(self, batch, batch_idx)` |
| Body of the validation loop | `validation_step(self, batch, batch_idx)` |
| Body of the test loop | `test_step(self, batch, batch_idx)` |
| Optimizer | `configure_optimizers(self)` |

Inside the steps we can record metrics using `self.log(name, value)`. Lightning will average them over the epoch, show them in the progress bar (if we ask for it with `prog_bar=True`) and send them to a **logger**.

Note that there are no calls to `.to(device)`, `zero_grad()`, `backward()`, `step()`, `net.train()`, `net.eval()` or `torch.no_grad()` - Lightning does all of this for us.

In [ ]:
class MNISTClassifier(pl.LightningModule):
    def __init__(self, hidden_size=128, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()   # remember hidden_size and lr (they get stored in logs and checkpoints)
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(28*28, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size, 10)
        )

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = torch.nn.functional.cross_entropy(self(x), y)
        self.log('train_loss', loss)
        return loss                   # Lightning will call backward() and the optimizer for us

    def evaluate(self, batch, stage):
        # common code for validation and test
        x, y = batch
        z = self(x)
        loss = torch.nn.functional.cross_entropy(z, y)
        acc = (z.argmax(dim=1) == y).float().mean()
        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc', acc, prog_bar=True)

    def validation_step(self, batch, batch_idx):
        self.evaluate(batch, 'val')

    def test_step(self, batch, batch_idx):
        self.evaluate(batch, 'test')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## Defining Data as a LightningDataModule

We already have our `DataLoader`s and could pass them directly to Lightning. However, in the code above the data preparation is scattered over several cells. Lightning offers `LightningDataModule` to keep all data-related code - downloading, splitting, transforms and data loaders - in one reusable class. It makes it easy to share the exact same data pipeline between experiments:

* `prepare_data` - download the data (called once)
* `setup` - split the data into train/validation/test datasets
* `train_dataloader`, `val_dataloader`, `test_dataloader` - return the corresponding `DataLoader`s

In [ ]:
class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir='data', batch_size=64):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.transform = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.1307,), (0.3081,))
        ])

    def prepare_data(self):
        torchvision.datasets.MNIST(self.data_dir, train=True, download=True)
        torchvision.datasets.MNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        full = torchvision.datasets.MNIST(self.data_dir, train=True, transform=self.transform)
        self.train_ds, self.val_ds = torch.utils.data.random_split(
            full, [55000, 5000], generator=torch.Generator().manual_seed(42))
        self.test_ds = torchvision.datasets.MNIST(self.data_dir, train=False, transform=self.transform)

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.val_ds, batch_size=self.batch_size)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.test_ds, batch_size=self.batch_size)

dm = MNISTDataModule()

## Training with the Trainer

The training is done by an object of the `Trainer` class, which "mixes" the model with the data and runs all the loops. Its most important parameters are:

* `max_epochs` - the number of epochs to train
* `accelerator` - the hardware to use. The value `'auto'` selects a GPU if there is one, and CPU otherwise (so we do not need the `device` variable anymore)
* `logger` - where to store the metrics that we record using `self.log`. We use `CSVLogger`, which simply writes them into a CSV file

Before the real training, it is a good habit to do a quick **smoke test** with `fast_dev_run=True`. In this mode the trainer runs just one batch of training, validation and test to make sure that there are no bugs in the code:

In [ ]:
trainer = pl.Trainer(fast_dev_run=True, accelerator='auto', devices=1)
trainer.fit(MNISTClassifier(), datamodule=dm)

Now let's train the model for real. The progress bar shows the loss and accuracy that we have asked for using `prog_bar=True`:

In [ ]:
model = MNISTClassifier(hidden_size=128, lr=1e-3)
logger = pl.loggers.CSVLogger('logs', name='mnist_mlp')
trainer = pl.Trainer(max_epochs=5, accelerator='auto', devices=1, logger=logger, log_every_n_steps=50)
trainer.fit(model, datamodule=dm)

## Looking at the Training Curves

Because we have logged the metrics using `self.log`, we do not need to collect them manually (it was Task 1 in the [Introduction to PyTorch](IntroPyTorch.ipynb)). The `CSVLogger` has saved them into a file, which we can read using `pandas`. Training metrics are recorded every 50 steps, and validation metrics once per epoch, so for each epoch we take the average of all values that were recorded:

In [ ]:
metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')
per_epoch = metrics.groupby('epoch').mean()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(per_epoch.index, per_epoch['train_loss'], label='train')
ax[0].plot(per_epoch.index, per_epoch['val_loss'], label='validation')
ax[0].set_xlabel('epoch'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(per_epoch.index, per_epoch['val_acc'], label='validation')
ax[1].set_xlabel('epoch'); ax[1].set_title('Accuracy'); ax[1].legend()
plt.show()

> If you have [TensorBoard](https://www.tensorflow.org/tensorboard) installed, you can use `pl.loggers.TensorBoardLogger` instead of `CSVLogger` and get interactive graphs without changing anything else in the code.

## Testing the Model

When we are done with experiments with the model, we measure its final quality on the test dataset, which the model has never seen. Lightning runs `test_step` for each batch of the test dataset and reports the averaged metrics:

In [ ]:
trainer.test(model, datamodule=dm)

## Making Predictions

`LightningModule` is a normal PyTorch module, so we can apply it to the data directly. Let's take a batch from the test dataset and look at the predictions. Since we are doing it ourselves outside of the `Trainer`, we need to take care of `eval()` and `no_grad()` as usual:

In [ ]:
x, y = next(iter(dm.test_dataloader()))

model.eval()
with torch.no_grad():
    pred = model(x.to(model.device)).argmax(dim=1).cpu()

fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    ax[i].imshow(x[i].squeeze(), cmap='gray')
    ax[i].set_title(f'{pred[i].item()}', color='green' if pred[i] == y[i] else 'red')
    ax[i].axis('off')
plt.show()

The trainer can also save the **checkpoints** of the model. By default, a checkpoint is saved to the log folder after each epoch, and we can restore the model later using `MNISTClassifier.load_from_checkpoint(path)`. Because we have used `save_hyperparameters()`, the values of `hidden_size` and `lr` are restored automatically:

In [ ]:
ckpt_path = trainer.checkpoint_callback.best_model_path
print('Checkpoint saved to:', ckpt_path)

restored = MNISTClassifier.load_from_checkpoint(ckpt_path)
print('Restored hyperparameters:', restored.hparams)

## What Did We Gain?

Let's compare the two approaches. Everything that was in the left column is still happening, but we no longer write it ourselves:

| Plain PyTorch | PyTorch Lightning |
|---------------|-------------------|
| `net.to(device)`, `x.to(device)` | automatic (`accelerator='auto'`) |
| `net.train()` / `net.eval()` / `torch.no_grad()` | automatic |
| `optim.zero_grad()`, `loss.backward()`, `optim.step()` | automatic, we just return the loss from `training_step` |
| Manual loops over epochs and minibatches | `trainer.fit(...)` |
| Accumulating loss and accuracy, printing them | `self.log(...)`, progress bar and loggers |
| Storing the metrics to plot them | `CSVLogger` / `TensorBoardLogger` |
| Writing a separate evaluation function for test data | `test_step` and `trainer.test(...)` |
| Saving / restoring models | checkpoints are saved automatically |
| Switching to multiple GPUs or TPUs needs code changes | `Trainer(accelerator=..., devices=...)` |

Lightning does not make the network better - it makes your code **shorter, more structured, and easier to reuse and share**.

> ✅ **Compare with [IntroPyTorch](IntroPyTorch.ipynb)**: look at the `train` function there (or at `train_pytorch` above) and at the `training_step`/`validation_step` methods here. Which lines of the plain PyTorch loop do you no longer see in the Lightning code? Where did they go? When would you still prefer to write the loop yourself?

## Tasks

**Task 1**: Change the hyperparameters of the model: train with different `hidden_size` (e.g. 32, 256, 512) and learning rate. Use a different `name` for the `CSVLogger` for each run, and compare the final validation accuracy. What is the best test accuracy you were able to achieve?

**Task 2**: Add one more hidden layer to the network. Does it help?

**Task 3**: Use a `pl.callbacks.EarlyStopping(monitor='val_loss')` callback by passing `callbacks=[...]` to the `Trainer` and set `max_epochs=30`. At which epoch did the training stop?

**Task 4**: Compare the Lightning code with the Keras version of the same problem in [Introduction to Keras](IntroKeras.ipynb). Which parts of the Keras `compile` / `fit` calls correspond to `configure_optimizers` and `Trainer` in Lightning?

## Takeaways

* **PyTorch Lightning** is a high-level API on top of PyTorch, which keeps the flexibility of PyTorch but removes the boilerplate of the training loop.
* The model goes into a `LightningModule` (`training_step`, `validation_step`, `test_step`, `configure_optimizers`), the data - into a `LightningDataModule` (or just plain `DataLoader`s), and the `Trainer` runs the training.
* Metrics are recorded with `self.log` and are available in the progress bar and in the loggers; devices and checkpoints are taken care of automatically.
* Since a `LightningModule` is a `torch.nn.Module`, you can use everything you know about PyTorch inside of it, for example, your own layers or the convolutional networks that we will study in the next lessons.